# Nested underspecified descriptions

`MoveToReach` nests a whole `GraspDescription` inside itself, and that `GraspDescription` holds an
`EndEffector` that must be selected from the world. A single query therefore mixes several kinds of
unknown at two levels of depth:

| level | slot | kind |
|---|---|---|
| outer | `MoveToReach.hip_rotation` | a value to generate |
| outer | `MoveToReach.target_pose_offset_robot.x` | a value inside a nested `Pose2D` |
| inner | `MoveToReach.grasp_description.approach_direction` | a value inside a nested designator |
| inner | `MoveToReach.grasp_description.end_effector` | an individual to select from the world |

This notebook checks what nesting does to reflection, the prompt, constraint enforcement and
construction. Everything runs offline with a `ScriptedClient`.

In [1]:
import json

from krrood.entity_query_language.factories import a, variable
from semantic_digital_twin.spatial_types import Pose, Pose2D

from coraplex.datastructures.grasp import GraspDescription
from coraplex.querying.language_model import (
    SituationCandidateRenderer,
    SituationContextProvider,
)
from coraplex.robot_plans.actions.core.misc import MoveToReach
from llm_reasoner.clients import ScriptedClient
from llm_reasoner.resolver import (
    LanguageModelBackend,
    NoValidCandidates,
    SlotKind,
    SourceDetail,
    reflect_slots,
)

## The world

Same builder as `language_model_backend.ipynb`. The apartment matters: a robot-only world contains
nothing manipulable, so the target and permitted-grasp sections would be correctly empty and this
notebook would demonstrate nothing about them.

In [2]:
import os

from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.robots.robot_parts import AbstractRobot, EndEffector
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Drawer,
    Handle,
    Milk,
    Spoon,
)
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.connections import (
    Connection6DoF,
    FixedConnection,
)
from semantic_digital_twin.world_description.world_entity import Body

from coraplex.querying.language_model import end_effectors_of

REPOSITORY_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
RESOURCES = os.path.join(REPOSITORY_ROOT, "coraplex", "resources")


def build_robot_world(robot_semantic_annotation=PR2) -> World:
    """Parse a robot from URDF and root it under map -> odom_combined."""
    world = URDFParser.from_file(
        file_path=robot_semantic_annotation.get_ros_file_path()
    ).parse()
    robot_semantic_annotation.from_world(world)

    with world.modify_world():
        old_root = world.root
        map_body = Body(name=PrefixedName("map"))
        localization_body = Body(name=PrefixedName("odom_combined"))
        world.add_connection(
            Connection6DoF.create_with_dofs(world, map_body, localization_body)
        )
        drive = robot_semantic_annotation.get_drive_connection_type().create_with_dofs(
            parent=localization_body, child=old_root, world=world
        )
        world.add_connection(drive)
        drive.has_hardware_interface = True

    return world


def build_apartment_world() -> World:
    """Parse the apartment and place annotated objects in it."""
    apartment = URDFParser.from_file(
        os.path.join(RESOURCES, "worlds", "apartment.urdf")
    ).parse()
    milk = STLParser(os.path.join(RESOURCES, "objects", "milk.stl")).parse()
    cereal = STLParser(
        os.path.join(RESOURCES, "objects", "breakfast_cereal.stl")
    ).parse()
    spoon = STLParser(os.path.join(RESOURCES, "objects", "spoon.stl")).parse()

    apartment.merge_world_at_pose(
        milk,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            2.37, 2, 1.05, reference_frame=apartment.root
        ),
    )
    apartment.merge_world_at_pose(
        cereal,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            2.37, 2.5, 1.05, reference_frame=apartment.root
        ),
    )
    apartment.merge_world(
        spoon,
        FixedConnection(
            parent=apartment.get_body_by_name("cabinet10_drawer_top"),
            child=spoon.root,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                -0.05, -0.05, -0.02
            ),
        ),
    )

    with apartment.modify_world():
        apartment.add_semantic_annotations(
            [
                Milk(root=apartment.get_body_by_name("milk.stl")),
                Spoon(root=apartment.get_body_by_name("spoon.stl")),
            ]
        )
        apartment.add_semantic_annotation_recursively(
            Drawer(
                root=apartment.get_body_by_name("cabinet10_drawer_top"),
                handle=Handle(root=apartment.get_body_by_name("handle_cab10_t")),
            )
        )

    return apartment


world = build_apartment_world()
world.merge_world(build_robot_world(PR2))

robot = world.get_semantic_annotations_by_type(AbstractRobot)[0]
end_effectors = end_effectors_of(robot)
milk = world.get_body_by_name("milk.stl")

print(robot.name.name, "->", [e.name.name for e in end_effectors])
print("target:", milk.name.name)

Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]


PR2 -> ['PR2LeftGripper', 'PR2RightGripper']
target: milk.stl


## The nested query

Two levels: a `Pose2D` and a whole `GraspDescription` nested inside a `MoveToReach`, with the end
effector selected from the world at the inner level.

`target_pose_end_effector` is bound to the **milk's actual pose**, not a placeholder. That is what
lets the bridge identify the target and compute which grasps its orientation permits -- with an
origin placeholder there is no object at that location and both sections stay empty.

> **Declare an extension at the type its candidates share.** `variable(T, domain)` *filters*
> `domain` by `T`. On a PR2 the grippers are `PR2LeftGripper` and `PR2RightGripper`, so
> `variable(type(end_effectors[0]), end_effectors)` would silently keep only the left one and a
> candidate token of `1` would be out of range. `EndEffector` is their common supertype.

> **Bind conditions through `match.variable`.** Conditions reach into the nesting:
> `match.variable.grasp_description.manipulation_offset` works. `match.expression` compiles,
> verbalizes correctly, and matches nothing -- silently, at any depth.

In [3]:
import rclpy
from rclpy.node import Node
import threading
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from coraplex.datastructures.dataclasses import Context

def start_ros_node(node_name: str) -> tuple[Node, threading.Thread]:
    """Start a ROS node and retain its background spinning thread."""
    if not rclpy.ok():
        rclpy.init()
    ros_node = rclpy.create_node(node_name)
    ros_thread = threading.Thread(target=rclpy.spin, args=(ros_node,), daemon=True)
    ros_thread.start()
    return ros_node, ros_thread


def start_visualization_publisher(
    world: World,
    ros_node: Node,
) -> tuple[TFPublisher, VizMarkerPublisher]:
    """Start synchronized transform and marker publication for a world."""
    tf_publisher = TFPublisher(_world=world, node=ros_node)
    visualization_publisher = VizMarkerPublisher(_world=world, node=ros_node)
    return tf_publisher, visualization_publisher

In [4]:
ros_node, ros_thread = start_ros_node("underspecified_action")
context = Context(world=world, robot=robot, ros_node=ros_node)

In [5]:
tf_publisher, visualization_publisher = start_visualization_publisher(
    world,
    ros_node,
)
print(f"RViz fixed frame: {world.root.name}")
print(f"MarkerArray topic: {visualization_publisher.topic_name}")


RViz fixed frame: apartment/apartment_root
MarkerArray topic: /semworld/viz_marker


In [6]:
def underspecified_move_to_reach():
    match = a(MoveToReach)(
        target_pose_offset_robot=a(Pose2D)(x=..., y=..., yaw=..., reference_frame=None),
        hip_rotation=...,
        target_pose_end_effector=milk.global_pose,
        grasp_description=a(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            end_effector=variable(EndEffector, end_effectors),
            rotate_gripper=...,
            manipulation_offset=...,
        ),
    )
    match.where(match.variable.grasp_description.manipulation_offset > 0.0)
    return match


backend = LanguageModelBackend(
    client=ScriptedClient(),
    context_provider=SituationContextProvider(world=world, robot=robot),
    candidate_renderer=SituationCandidateRenderer(world=world, robot=robot),
)

## Reflection flattens the tree

Nesting becomes dotted paths, and the nesting node itself is **not** a slot: there is no entry for
`MoveToReach.grasp_description`, only for its leaves. That is what lets one prompt and one
write-back cover a whole designator tree with no special handling for depth.

Traversal is breadth-first, so the outer object's leaves come first regardless of declaration
order. `owner` is the class that *declares* each attribute, which is how the right docstring is
found at any depth.

In [7]:
for slot in reflect_slots(underspecified_move_to_reach()):
    detail = ""
    if slot.members:
        detail = " | ".join(member.name for member in slot.members)
    elif slot.domain is not None:
        detail = f"{len(slot.domain)} candidate(s)"
    print(
        f"{slot.kind.name:9} {slot.path:52} "
        f"owner={slot.owner.__name__:18} {detail}"
    )

VALUE     MoveToReach.hip_rotation                             owner=MoveToReach        
BOUND     MoveToReach.target_pose_end_effector                 owner=MoveToReach        
VALUE     MoveToReach.target_pose_offset_robot.x               owner=Pose2D             
VALUE     MoveToReach.target_pose_offset_robot.y               owner=Pose2D             
VALUE     MoveToReach.target_pose_offset_robot.yaw             owner=Pose2D             
BOUND     MoveToReach.target_pose_offset_robot.reference_frame owner=Pose2D             
VALUE     MoveToReach.grasp_description.approach_direction     owner=GraspDescription   FRONT | BACK | RIGHT | LEFT
VALUE     MoveToReach.grasp_description.vertical_alignment     owner=GraspDescription   NoAlignment | TOP | BOTTOM
REFERENT  MoveToReach.grasp_description.end_effector           owner=GraspDescription   2 candidate(s)
VALUE     MoveToReach.grasp_description.rotate_gripper         owner=GraspDescription   
VALUE     MoveToReach.grasp_description.man

### Bound slots

`target_pose_end_effector` holds a concrete `Pose`, so it is `BOUND`: it appears under *Already
known* as context and is never asked for. Depth does not change that -- a bound slot inside a
nested designator behaves the same way.

In [8]:
bound = [
    slot
    for slot in reflect_slots(underspecified_move_to_reach())
    if slot.kind is SlotKind.BOUND
]
[(slot.path, type(slot.bound_value).__name__) for slot in bound]

[('MoveToReach.target_pose_end_effector', 'Pose'),
 ('MoveToReach.target_pose_offset_robot.reference_frame', 'NoneType')]

## The prompt

In [9]:
print(backend.describe(underspecified_move_to_reach()))

Generate 3 candidate MoveToReach object(s).

## What these are

**MoveToReach** -- Let the robot move to a position facing the target and reach with a end_effector.

**Pose2D** -- Represents a 2D pose consisting of an x coordinate, a y coordinate, and a yaw angle.

Internally stored as a 3×1 symbolic vector ``[x, y, yaw]``. Behaves similarly to
:class:`Pose`, but lives in the 2D plane (z=0, roll=0, pitch=0). Whenever 3D
calculations are required, use :meth:`to_pose` to obtain the equivalent 3D
:class:`Pose`.

**GraspDescription** -- Describes a grasp configuration for a end_effector the description consists of the
approach direction (the side from which to grasp e.g. FRONT, LEFT, etc and the
vertical alignment (TOP, BOTTOM).

## The request

Generate a MoveToReach whose hip_rotation is Ellipsis, and whose target_pose_end_effector is a specific Pose, such that the x of its target_pose_offset_robot is Ellipsis, the y of its target_pose_offset_robot is Ellipsis, the yaw of its target_pose

### Every nested class explains itself

`SourceDetail.SIGNATURE` emits a docstring for the constructed type **and** for every other class
declaring a slot. Without that, a nested query would tell the model what a `MoveToReach` is and
leave it to infer `GraspDescription` and `Pose2D` from field names.

In [10]:
prompt = backend.describe(underspecified_move_to_reach())

for target_type in (MoveToReach, GraspDescription, Pose2D):
    marker = (target_type.__doc__ or "").strip().split("\n")[0][:40]
    print(f"{target_type.__name__:20} described: {marker in prompt}")

print()
print(prompt.split("## What these are")[1].split("## The request")[0].strip())

MoveToReach          described: True
GraspDescription     described: True
Pose2D               described: True

**MoveToReach** -- Let the robot move to a position facing the target and reach with a end_effector.

**Pose2D** -- Represents a 2D pose consisting of an x coordinate, a y coordinate, and a yaw angle.

Internally stored as a 3×1 symbolic vector ``[x, y, yaw]``. Behaves similarly to
:class:`Pose`, but lives in the 2D plane (z=0, roll=0, pitch=0). Whenever 3D
calculations are required, use :meth:`to_pose` to obtain the equivalent 3D
:class:`Pose`.

**GraspDescription** -- Describes a grasp configuration for a end_effector the description consists of the
approach direction (the side from which to grasp e.g. FRONT, LEFT, etc and the
vertical alignment (TOP, BOTTOM).


### The target is recovered from a bound pose

`object_designator` is not a field of `MoveToReach`, so the bridge resolves the target from the
bound `target_pose_end_effector` instead: the nearest body with collision geometry, excluding the
robot's own links. Frames such as `map` and `odom_combined` have no geometry and are never
candidates -- naming one as the target would be confidently wrong, which is worse than saying
nothing.

Once the target is known, `GraspDescription.calculate_grasp_descriptions` reports which approach
directions and vertical alignments the object's orientation actually permits.

In [11]:
from coraplex.querying.language_model import target_body_of, bodies_of

target = target_body_of(
    world, underspecified_move_to_reach(), excluded=bodies_of(robot)
)
print("target:", target.name.name if target else None, "| expected: milk.stl")

situation_text = SituationContextProvider(world=world, robot=robot).context_for(
    underspecified_move_to_reach()
)
print()
print(situation_text)

target: milk.stl | expected: milk.stl

The object being acted on:
- milk.stl  a Milk  0.06 x 0.07 x 0.19 m (depth x width x height)  on apartment_root  2.37 m ahead, 2.00 m to the left, 1.05 m above of base_footprint

Grasps permitted by how milk.stl is turned relative to the robot.
Approach directions name faces of the object, not of the robot, and the first listed is the face most turned toward the robot:
- approach_direction FRONT, vertical_alignment NoAlignment
- approach_direction RIGHT, vertical_alignment NoAlignment
- approach_direction FRONT, vertical_alignment TOP
- approach_direction RIGHT, vertical_alignment TOP
- approach_direction FRONT, vertical_alignment BOTTOM
- approach_direction RIGHT, vertical_alignment BOTTOM

Positions below are given relative to PR2, which faces along its own forward axis.
Its end effectors:
- PR2LeftGripper: free
- PR2RightGripper: free

Other objects in the scene:
- spoon.stl  a Spoon  0.22 x 0.05 x 0.02 m (depth x width x height)  on cabinet10_

## Constraints reach into the nesting

The condition was written against `match.variable.grasp_description.manipulation_offset`. It is
stated in the prompt *and* applied afterwards as a filter, so a nested violation is caught even
though the model was told not to produce one.

In [12]:
def response(manipulation_offset=0.06, end_effector_token=0):
    return json.dumps([
        {
            "MoveToReach.target_pose_offset_robot.x": 0.4,
            "MoveToReach.target_pose_offset_robot.y": 0.0,
            "MoveToReach.target_pose_offset_robot.yaw": 0.0,
            "MoveToReach.hip_rotation": 0.0,
            "MoveToReach.grasp_description.approach_direction": "FRONT",
            "MoveToReach.grasp_description.vertical_alignment": "TOP",
            "MoveToReach.grasp_description.end_effector": end_effector_token,
            "MoveToReach.grasp_description.rotate_gripper": False,
            "MoveToReach.grasp_description.manipulation_offset": manipulation_offset,
        }
    ])


def resolve(raw):
    return LanguageModelBackend(
        client=ScriptedClient(responses=[raw]),
        context_provider=SituationContextProvider(world=world, robot=robot),
        candidate_renderer=SituationCandidateRenderer(world=world, robot=robot),
        number_of_candidates=1,
    ).evaluate(underspecified_move_to_reach())


try:
    list(resolve(response(manipulation_offset=-0.02)))
except NoValidCandidates as error:
    print("nested constraint enforced:", error)

nested constraint enforced: All 1 constructed candidate(s) failed the query's where-conditions.


## Construction recurses

Write-back walks each leaf's access path into the nested kwargs dictionaries, then
`construct_instance` calls the factory tree: `Pose2D(**kwargs)` and `GraspDescription(**kwargs)`
are built first and passed into `MoveToReach(**kwargs)`.

All three classes were supplied by the caller in the query. Nothing in `llm_reasoner` imports any
of them, which is why depth costs nothing.

In [13]:
action = next(iter(resolve(response(end_effector_token=1))))

print(type(action).__name__)
print("  target_pose_offset_robot:", type(action.target_pose_offset_robot).__name__,
      action.target_pose_offset_robot.x, action.target_pose_offset_robot.y)
print("  grasp_description:       ", type(action.grasp_description).__name__)
print("  end_effector:            ", action.grasp_description.end_effector.name.name)
print("  is the live world object:", action.grasp_description.end_effector is end_effectors[1])

MoveToReach
  target_pose_offset_robot: Pose2D 0.4 0
  grasp_description:        GraspDescription
  end_effector:             PR2RightGripper
  is the live world object: True


### Using Real LLM Client

In [14]:
from dotenv import load_dotenv
load_dotenv(override=True)

from llm_reasoner.clients import AnthropicClient

live = LanguageModelBackend(
    client=AnthropicClient(),
    context_provider=SituationContextProvider(world=world, robot=robot),
    candidate_renderer=SituationCandidateRenderer(world=world, robot=robot),
    number_of_candidates=3,
)

# for action in live.evaluate(underspecified_move_to_reach()):
#     print(action)
#     print(type(action))
#
# print(live.log[-1].raw_response)

In [15]:
resolved_action = live.evaluate(underspecified_move_to_reach())
resolved_action = next(iter(resolved_action))

In [16]:
type(resolved_action)

coraplex.robot_plans.actions.core.misc.MoveToReach

### Execute the Resolved Action as part of the plan

In [17]:
from coraplex.execution_environment import simulated_robot
from coraplex.plans.factories import code, sequential
from coraplex.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from semantic_digital_twin.datastructures.definitions import TorsoState
from coraplex.datastructures.enums import Arms, ApproachDirection, VerticalAlignment

task = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        resolved_action
    ],
    context=context,
)

with simulated_robot:
    task.perform()